In [1]:
# ==============================================================================
# PIPELINE 3: Lot-Level Matching & Structural Integrity Analysis
# ==============================================================================
import pandas as pd
from src.profiling import generate_minimal_report
from src.preparation.matching import split_by_lot, finalize_datasets
from src.preparation.match_analysis import (
    compare_metrics, 
    analyze_cross_dataset_orphans, 
    analyze_partial_notice_success
)

pd.set_option('display.max_columns', None)

# ------------------------------------------------------------------------------
# STEP 1: Load Notice-Matched Data from Pipeline 2
# ------------------------------------------------------------------------------
print("Loading notice-matched datasets...")
df_cfc_notices = pd.read_parquet("data/prepared/CFC_2018_2023.parquet")
df_can_notices = pd.read_parquet("data/prepared/CAN_2018_2023.parquet")

Loading notice-matched datasets...


In [2]:
# ------------------------------------------------------------------------------
# STEP 2: Lot-Level Split (Strict Composite Key Match)
# ------------------------------------------------------------------------------
# Matches are identified via 'FUTURE_CAN_ID' + 'ID_LOT'
split_results = split_by_lot(df_cfc_notices, df_can_notices)

cfc_matched = split_results['cfc_matched']
cfc_orphans = split_results['cfc_orphans']
can_matched = split_results['can_matched']
can_orphans = split_results['can_orphans']

 -> Lot-Level Match: 4,210,162 strict composite keys aligned.
    CFC: 4,210,390 matches | 2,098,611 orphans.
    CAN: 4,622,774 matches | 221,503 orphans.


In [3]:
# ------------------------------------------------------------------------------
# STEP 3: Structural Integrity Analysis
# ------------------------------------------------------------------------------

# 3.1 Cross-Dataset Orphans (Missing lot definitions in matched notices)
print("Analyzing cross-dataset structural orphans...")
display(analyze_cross_dataset_orphans(cfc_orphans, can_orphans))

# 3.2 Partial Success Analysis (Total vs. Partial failure of notice lots)
print("\nAnalyzing partial vs. total failure scenarios for CFC notices...")
display(analyze_partial_notice_success(cfc_matched, cfc_orphans))

Analyzing cross-dataset structural orphans...
 -> Identified cross-dataset structural orphans.


,Direction,Affected Notices
0,CFC Lot Orphan in Matched CAN,33215
1,CAN Lot Orphan in Matched CFC,33215



Analyzing partial vs. total failure scenarios for CFC notices...
 -> Analyzed partial vs. total failure scenarios for orphaned notices.


,Scenario,Notice Count,Share (%)
0,Total Failure (0% Lots Awarded),28119,20.7
1,Partial Success (>0% Lots Awarded),107754,79.3


In [4]:
# ------------------------------------------------------------------------------
# STEP 4: Economic Bias Analysis
# ------------------------------------------------------------------------------
# Checking if lot-level orphans are economically different from matched lots
print("Comparing financial metrics for CFC lots (Matched vs. Orphans):")
display(compare_metrics(cfc_matched, cfc_orphans, metrics=['TOTAL_VALUE_EUR', 'DURATION']))

print("\nComparing financial metrics for CAN lots (Matched vs. Orphans):")
display(compare_metrics(can_matched, can_orphans, metrics=['TOTAL_VALUE_EUR', 'LOT_AWARD_VALUE_EUR']))

Comparing financial metrics for CFC lots (Matched vs. Orphans):
 -> Compared central tendencies for 2 metric(s).


,Metric,Matched_Median,Orphan_Median,Matched_Mean,Orphan_Mean
0,TOTAL_VALUE_EUR,1041073.92,1689642.91,49549568.82,29476029.44
1,DURATION,24.00,24.00,24.62,24.97



Comparing financial metrics for CAN lots (Matched vs. Orphans):
 -> Compared central tendencies for 2 metric(s).


,Metric,Matched_Median,Orphan_Median,Matched_Mean,Orphan_Mean
0,TOTAL_VALUE_EUR,363117.98,468743.88,1.448163e+07,2.212839e+07
1,LOT_AWARD_VALUE_EUR,18309.47,21025.43,1.629834e+08,1.765632e+08


In [5]:
# ------------------------------------------------------------------------------
# STEP 5: Finalize & Save Checkpoints
# ------------------------------------------------------------------------------
df_cfc_lots, df_can_lots = finalize_datasets(split_results)

df_cfc_lots.to_parquet("data/prepared/CFC_2018_2023.parquet", index=False)
df_can_lots.to_parquet("data/prepared/CAN_2018_2023.parquet", index=False)

# Generate a minimal report for the full CFC data
generate_minimal_report(
    df=df_cfc_lots,
    output_path="reports/processed/cfc_minimal_lots.html",
    title="CFC Lots Matching (minimal; full data)"
)

# Generate a minimal report for the full CAN data
generate_minimal_report(
    df=df_can_lots,
    output_path="reports/processed/can_minimal_lots.html",
    title="CAN Lots Matching (minimal; full data)"
)

 -> Finalized datasets. Orphans discarded.
    CFC Final Rows: 4,210,390
    CAN Final Rows: 4,622,774
Generating minimal report: 'CFC Lots Matching (minimal; full data)' ...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 26/26 [00:03<00:00,  8.52it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 31/31 [00:02<00:00, 14.27it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

-> Minimal report successfully saved to: reports/processed/can_minimal_lots.html

